# **Part I: Data Preprocessing**

## Goal: Understand how to prepare data for analysis. Familiarize with the World Development Indicators dataset and reshape it into panel data.

> Path note: this notebook now looks for files relative to the repository. In Colab, clone or upload the repository before running the code.


# **World Development Indicators**

>https://datatopics.worldbank.org/world-development-indicators/
<img src="Figures/WDIfigure.jpg" alt="localImage" style="width: 1200px; height: auto;" />

## Panel data, also known as longitudinal data or cross-sectional time series data, is a type of data that combines both cross-sectional and time series dimensions. 
### What does panel data looks like?

<img src="Figures/paneldata.jpg" alt="localImage" style="width: 600px; height: auto;" />


# Q1 How to read the CSV file and name the dataframe as df?

In [ ]:
import pandas as pd
from pathlib import Path


def locate_first(*relative_path_options):
    current = Path.cwd().resolve()
    search_roots = [current, *current.parents]
    for parts in relative_path_options:
        for base in search_roots:
            candidate = base.joinpath(*parts)
            if candidate.exists():
                return candidate
    options = ", ".join("/".join(parts) for parts in relative_path_options)
    raise FileNotFoundError(
        f"Could not locate any of these files: {options}. Please place the WDI CSV file in the repository Data folder first."
    )


OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(
    locate_first(
        ("archive/legacy_wdi_materials", "Data", "WDICSV.csv"),
        ("Data", "WDICSV.csv"),
    )
)


In [ ]:
### Check the first or last few rows.
## Check the first 5 rows.
df.head()

In [ ]:
## Check the last 3 rows.
df.tail(3)

# Q2 What is the type of the DataFrame?

## In a pandas DataFrame, there are several data types that you can encounter. The main types of data include:


### 1. Float: Decimal numbers (e.g., float32, float64).
### 2. Object: Used for strings or mixed types. This is the default type for text data.
### 3. Integer: Whole numbers, which can be of different sizes (e.g., int32, int64).
### 4. Boolean: Represents True or False values.
### 5. Datetime: Used for date and time data (e.g., datetime64).
### 6. Timedelta: Represents differences in time (duration).

# Q3 How many columns and rows?

In [ ]:
df.info()

# Q4 What are the country names in this dataset?

In [ ]:
unique_countries = df["Country Name"].unique()
unique_countries

print(', '.join(unique_countries))

# Q5 How many indicators and countries?

In [ ]:
## Use nuique() to get the number of unique values.
df["Country Name"].nunique()
### There are 266 countries in the WDI dataset.

In [ ]:
df["Indicator Name"].nunique()
### WDI dataset provides 1516 indicators.

# Q6 How to drop and keep certain columns?

In [ ]:
df.head(3)

In [ ]:
###Drop the last column
df.drop(columns=["2024"])

In [ ]:
## Keep the certain columns
### For instance we want to keep the following three columns.
df[["Country Name", "Indicator Name", "2020"]]

# Q7 How to drop and keep certain rows?

In [ ]:
## Delete specific rows based on the index

### In Pandas, an index is a fundamental component that 
### serves as a unique identifier for each row in a DataFrame or Series.

### In this dataset, the index ranges from 0 to 403255. 
### We can drop the first row by specifying index[0].

df.drop(df.index[0])

In [ ]:
## Keep specific rows based on certain conditions
### Choose the country named United States
df_US = df[df["Country Name"] == "United States"]
df_US

In [ ]:
## How about two conditions? Use "&" to add more conditions.
df_US_gdp = df[
    (df["Country Name"] == "United States")
    & (df["Indicator Name"] == "GDP per capita (current US$)")
]

df_US_gdp

# Q8 How to reshape data?
## Pandas provides multiple methods like melt(), pivot_table(), stack(), unstack() ,etc to reshape data.
### https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html

## 1) melt(). This function is used to transform or reshape data from a wide format to a long format. It essentially unpivots the DataFrame, converting columns into rows.

### Wide Format:

#### Each variable is a separate column.

#### Example:

| Country Name | 1960 | 1961 | 1962 |
|--------------|------|------|------|
| USA          | 1000 | 1200 | 1400 |
| Canada       | 800  | 850  | 900  |

### Long Format:

#### Each observation is a separate row, with variables in two columns: one for variable names and one for values.
#### Example:

| Country Name | Year | GDP |
|--------------|------|-------|
| USA          | 1960 | 1000  |
| Canada       | 1960 | 800   |

### https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.melt.html#pandas.DataFrame.melt

### Key parameters:
#### id_vars: A list or tuple of column names to use as identifier variables. These columns will remain as columns in the resulting DataFrame.
#### value_vars: A list or tuple of column names to unpivot. These columns will be converted into a single column in the resulting DataFrame.
#### var_name: The name to use for the column that contains the variable names (default is 'variable').
#### value_name: The name to use for the column that contains the values (default is 'value').

In [ ]:
df_gdp_melt = df_US_gdp.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="GDP per capita (current US$)",
).drop(columns=["Country Code", "Indicator Code"])
df_gdp_melt

### By using the melt() function, the DataFrame df_gdp_melt is now clean and in panel format. 
### It contains unique values for Country Name and Year.

### In the last example, we used df_US_gdp, which contains only GDP per capita (current US$) for the United States. It could become more complicated if we add more variables. For instance, we can use df_US, which contains all the variables provided by the WDI.

In [ ]:
df_US

In [ ]:
df_melt = df_US.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
).drop(columns=["Country Code", "Indicator Code"])

### df_melt isn't a panel data we wanted, we want the Indicator Name listed by columns.
df_melt

## 2） pivot_table(). This function is used to create a pivot table from a DataFrame. It allows you to summarize and aggregate data based on one or more columns, providing insights into the relationships between different variables.
### https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html#pandas.pivot_table

### key parameters:

#### data: The DataFrame to be used for creating the pivot table.
#### values: The column(s) to aggregate.
#### index: The column(s) to be used as the index of the resulting pivot table.
#### columns: The column(s) to be used as the columns of the resulting pivot table.
#### aggfunc: The aggregation function(s) to apply to the values. It can be a single function, a list of functions, or a dictionary mapping columns to functions.
#### fill_value: The value to replace missing values with (default is None).

In [ ]:
df_pivottable = df_melt.pivot_table(
    values="value",
    index=["Country Name", "Year"],
    columns="Indicator Name",
)

df_pivottable.head()

### Now the variables in the Indicator Name are listed as columns. 
### However, it looks strange due to the multilevel index.

In [ ]:
# How to get rid of multilevel index after using pivottable?

# After using reset_index, we can get rid of the multilevel index.
WDI_US_0 = df_pivottable.reset_index()

WDI_US_0

In [ ]:
### However, in the new index, the first row named 'Indicator Name' should be empty. 
### Therefore, we use rename_axis to reset the index.
WDI_US = WDI_US_0.rename_axis("", axis=1)

WDI_US.head()

# Now, WDI_US is the panel data we wanted!

In [ ]:
### Recheck the data types and find that the column 'Year' is of type object.
### However, an object type cannot be used in mathematical calculations or sorting.
WDI_US.dtypes

In [ ]:
### So, the 'Year' column needs to be converted to an integer.
WDI_US["Year"] = WDI_US["Year"].astype(int)

WDI_US.dtypes

# Q9 How to export data to csv?

In [ ]:
WDI_US.to_csv(OUTPUT_DIR / "WDI_US.csv", index=False)


# Q10 How many missing values for each variable?
### isna(): This method checks each element in the WDI_US DataFrame for missing values (NaN). It returns a DataFrame of the same shape, where each element is either True (if the original value is NaN) or False (if the original value is not NaN).
### sum(): After checking for missing values, this method aggregates the results. By default, it sums along the columns (i.e., it counts the number of True values for each column). The result is a Series where the index is the column names of WDI_US, and the values are the counts of missing values (NaN) in each column.
### sort_values(ascending=True): This sorts the resulting Series by the count of missing values in ascending order. This means that columns with fewer missing values will appear first, and columns with more missing values will appear later.

In [ ]:
WDI_US.isna()

In [ ]:
WDI_US.isna().sum()

In [ ]:
isna_data = WDI_US.isna().sum().sort_values(ascending=True)
isna_data

# **Panel data get!**
## Now we obtain the data for the United States and all variables provided by the WDI dataset, reshape the data into panel format, and count the number of missing values for all variables.